# 🔭 CCD Photometric Reduction Pipeline (PyRAF / IRAF)
### Unified notebook — handles both 3-D FITS cubes *and* pre-unpacked science frames

**Pipeline steps:**
1. ⚙️  Configuration  
2. 📦  Package setup & output-directory creation  
3. 🔪  (Optional) Unpack 3-D FITS cubes with `imslice`  
4. 🧮  Master Bias creation  
5. 🔧  Bias-correction → Science frames  
6. 🔧  Bias-correction → Flat frames  
7. 🎛️  Master Flat creation  
8. 📐  Master Flat normalisation  
9. ✨  Flat-field correction → Science frames  

---

##  Cell 1 — Configuration
> **Edit only this cell.** All other cells read from these variables.

In [ ]:
# ╔══════════════════════════════════════════════════════════════╗
# ║              USER CONFIGURATION — EDIT HERE ONLY            ║
# ╚══════════════════════════════════════════════════════════════╝

import os

# ── 1. DATA LOCATIONS ──────────────────────────────────────────
# Directory that contains ALL your raw .fits files.
# Use '.' for the current working directory.
RAW_DATA_DIR = '.'

# ── 2. OUTPUT DIRECTORY ROOT ───────────────────────────────────
# Sub-folders  bias/ | flats/ | science/  will be created here.
OUTPUT_DIR = './reduced'

# ── 3. INPUT MODE ──────────────────────────────────────────────
# 'cube'      → raw files are 3-D FITS cubes; imslice will unpack them.
# 'unstacked' → files are already individual 2-D science frames.
SCIENCE_INPUT_MODE = 'unstacked'    # <-- change to 'unstacked' if already sliced

# ── 4. FILE GLOB PATTERNS ──────────────────────────────────────
# Patterns are relative to RAW_DATA_DIR.
BIAS_PATTERN       = '*bias*.fit'      # raw bias frames
FLAT_PATTERN       = '*flat*.fit'      # raw flat frames
# For cubes: pattern that selects the 3-D cube files to slice
CUBE_PATTERN       = '*01_12.fit'
# For unstacked: pattern that selects the already-sliced science frames
UNSTACKED_PATTERN  = '*GRB260421B_r_300s_*.fit'

# ── 5. INSTRUMENT / DETECTOR ───────────────────────────────────
RDNOISE  = 5.0    # e⁻  — read noise of your detector
GAIN     = 10.0   # e⁻/ADU — gain of your detector

# ── 6. ZEROCOMBINE PARAMETERS ──────────────────────────────────
BIAS_COMBINE  = 'average'   # 'average' | 'median' | 'sum'
BIAS_REJECT   = 'minmax'    # 'minmax'  | 'sigclip' | 'avsigclip'
BIAS_NLOW     = 0
BIAS_NHIGH    = 1

# ── 7. FLATCOMBINE PARAMETERS ──────────────────────────────────
FLAT_COMBINE  = 'median'    # 'median'  | 'average' | 'sum'
FLAT_REJECT   = 'avsigclip' # 'avsigclip' | 'sigclip' | 'minmax'
FLAT_LSIGMA   = 3.0
FLAT_HSIGMA   = 3.0
FLAT_SCALE    = 'mode'      # 'mode' | 'mean' | 'median' | 'none'

# ── 8. FILE PREFIX CONVENTIONS ─────────────────────────────────
PREFIX_BIAS_CORR  = 'b_'    # bias-subtracted files
PREFIX_FLAT_CORR  = 'f'     # flat-fielded files  (stacks with PREFIX_BIAS_CORR → fb_)

# ── 9. OUTPUT FILE NAMES ───────────────────────────────────────
MASTER_BIAS_NAME      = 'MasterBias.fits'
MASTER_FLAT_NAME      = 'MasterFlat.fits'
NORM_MASTER_FLAT_NAME = 'NormMasterFlat.fits'

# ── 10. IMSLICE SETTINGS (cube mode only) ──────────────────────
SLICE_DIMENSION = 3   # 3 = slice along Z-axis (standard for data cubes)


# ── 10. EDGE TRIMMING PARAMETERS (Pre-Alignment) ───────────────
TRIM_INPUT_WILDCARD = "fb_*.fits"
TRIM_PIXELS         = 10   # Number of pixels to cut off EVERY edge
TRIM_OUTPUT_PREFIX  = "t"

# ╔══════════════════════════════════════════════════════════════╗
# ║                 END OF USER CONFIGURATION                    ║
# ╚══════════════════════════════════════════════════════════════╝

# ── Derived absolute paths (do NOT edit below this line) ───────
RAW_DATA_DIR = os.path.abspath(RAW_DATA_DIR)

DIR_BIAS    = os.path.abspath(os.path.join(OUTPUT_DIR, 'bias'))
DIR_FLATS   = os.path.abspath(os.path.join(OUTPUT_DIR, 'flats'))
DIR_SCIENCE = os.path.abspath(os.path.join(OUTPUT_DIR, 'science'))

MASTER_BIAS_PATH      = os.path.join(DIR_BIAS,  MASTER_BIAS_NAME)
MASTER_FLAT_PATH      = os.path.join(DIR_FLATS, MASTER_FLAT_NAME)
NORM_MASTER_FLAT_PATH = os.path.join(DIR_FLATS, NORM_MASTER_FLAT_NAME)

print("✅ Configuration loaded.")
print(f"   RAW_DATA_DIR  : {RAW_DATA_DIR}")
print(f"   OUTPUT_DIR    : {os.path.abspath(OUTPUT_DIR)}")
print(f"   Mode          : {SCIENCE_INPUT_MODE}")
print(f"   Bias dir      : {DIR_BIAS}")
print(f"   Flats dir     : {DIR_FLATS}")
print(f"   Science dir   : {DIR_SCIENCE}")


✅ Configuration loaded.
   RAW_DATA_DIR  : /home/void/GRB_Data/GRB260421B
   OUTPUT_DIR    : /home/void/GRB_Data/GRB260421B/reduced
   Mode          : unstacked
   Bias dir      : /home/void/GRB_Data/GRB260421B/reduced/bias
   Flats dir     : /home/void/GRB_Data/GRB260421B/reduced/flats
   Science dir   : /home/void/GRB_Data/GRB260421B/reduced/science


##  Cell 2 — Package Setup & Output Directories

In [2]:
import glob
import shutil
from pyraf import iraf

# ── Load IRAF packages ─────────────────────────────────────────
iraf.noao()
iraf.imred()
iraf.ccdred()
iraf.images()
iraf.imutil()
print("✅ IRAF packages loaded.")

# ── Create output directories ──────────────────────────────────
for label, path in [('bias', DIR_BIAS), ('flats', DIR_FLATS), ('science', DIR_SCIENCE)]:
    os.makedirs(path, exist_ok=True)
    print(f"   📁 {label:8s} → {path}")

print("\n✅ Output directories ready.")

# ── Helper: resolve glob relative to RAW_DATA_DIR ──────────────
def find_files(pattern, base=RAW_DATA_DIR):
    """Glob *pattern* inside *base* and return a sorted list of absolute paths."""
    results = sorted(glob.glob(os.path.join(base, pattern)))
    return results

# ── Helper: write an IRAF @-list file ──────────────────────────
def write_list(paths, list_path):
    """Write *paths* (one per line) to *list_path* for IRAF @-file syntax."""
    with open(list_path, 'w') as fh:
        for p in paths:
            fh.write(p + '\n')
    print(f"   📝 List written: {list_path}  ({len(paths)} entries)")


Created directory /home/void/GRB_Data/GRB260421B/pyraf for cache
imred/:
 argus/         ctioslit/       hydra/          kpnocoude/      vtel/
 bias/          dtoi/           iids/           kpnoslit/
 ccdred/        echelle/        irred/          quadred/
 crutil/        generic/        irs/            specred/
ccdred/:
 badpiximage    ccdlist         combine         mkillumcor      setinstrument
 ccdgroups      ccdmask         darkcombine     mkillumflat     zerocombine
 ccdhedit       ccdproc         flatcombine     mkskycor
 ccdinstrument  ccdtest/        mkfringecor     mkskyflat
✅ IRAF packages loaded.
   📁 bias     → /home/void/GRB_Data/GRB260421B/reduced/bias
   📁 flats    → /home/void/GRB_Data/GRB260421B/reduced/flats
   📁 science  → /home/void/GRB_Data/GRB260421B/reduced/science

✅ Output directories ready.


##  Cell 3 — (Optional) Unpack 3-D FITS Cubes with `imslice`
> **Skipped automatically** when `SCIENCE_INPUT_MODE = 'unstacked'`.

In [3]:
# This cell only runs when SCIENCE_INPUT_MODE == 'cube'.
# Sliced frames are written directly into DIR_SCIENCE so that
# subsequent cells always look in the same place regardless of mode.

if SCIENCE_INPUT_MODE == 'cube':
    cube_files = find_files(CUBE_PATTERN)
    if not cube_files:
        print(f"⚠️  No cube files matched pattern '{CUBE_PATTERN}' in {RAW_DATA_DIR}")
    else:
        print(f"Found {len(cube_files)} cube file(s) to slice:\n")
        for cube in cube_files:
            # e.g.  /data/01_12.fits  →  <DIR_SCIENCE>/01_12_slice
            base     = os.path.splitext(os.path.basename(cube))[0]
            out_root = os.path.join(DIR_SCIENCE, base + '_slice')
            print(f"  Slicing  {os.path.basename(cube)}  →  {out_root}NNN.fits")
            try:
                iraf.imslice(
                    input          = cube,
                    output         = out_root,
                    slice_dimension= SLICE_DIMENSION,
                    verbose        = 'yes',
                    mode           = 'h'
                )
            except iraf.IrafError as exc:
                print(f"  ⚠️  imslice failed for {cube}:\n     {exc}")
        print("\n✅ All cubes sliced into:", DIR_SCIENCE)

elif SCIENCE_INPUT_MODE == 'unstacked':
    print("ℹ️  Mode = 'unstacked' — imslice step skipped.")
    print("   Pre-sliced science frames will be read from RAW_DATA_DIR in the next steps.")
else:
    raise ValueError(f"SCIENCE_INPUT_MODE must be 'cube' or 'unstacked', got '{SCIENCE_INPUT_MODE}'")


ℹ️  Mode = 'unstacked' — imslice step skipped.
   Pre-sliced science frames will be read from RAW_DATA_DIR in the next steps.


In [4]:
import os

RAW_DATA_DIR = '.'

# 1. Print the exact absolute path so you know precisely where Python is looking
absolute_path = os.path.abspath(RAW_DATA_DIR)
print(f"Python is currently looking inside:\n{absolute_path}\n")

# 2. Check for any files at all
all_files = os.listdir(RAW_DATA_DIR)
print(f"Total files/folders found here: {len(all_files)}")

# 3. Filter for just the .fits files
fits_files = [f for f in all_files if f.endswith('.fits')]

if not fits_files:
    print("\n⚠️ WARNING: Python sees this folder, but found ZERO .fits files directly inside it.")
    print("If your files are inside subfolders (like 'Dark&Bias'), you need to use the recursive search.")
else:
    print(f"\n✅ SUCCESS: Found {len(fits_files)} .fits files! Here are the first few:")
    # Print just the first 10 so it doesn't flood your screen
    for f in fits_files[:10]:
        print(f"  - {f}")

Python is currently looking inside:
/home/void/GRB_Data/GRB260421B

Total files/folders found here: 29

⚠️ WARNING: Python sees this folder, but found ZERO .fits files directly inside it.
If your files are inside subfolders (like 'Dark&Bias'), you need to use the recursive search.


##  Cell 4 — Master Bias Creation

In [5]:
bias_files = find_files(BIAS_PATTERN)

if not bias_files:
    raise FileNotFoundError(
        f"No bias frames found matching '{BIAS_PATTERN}' in {RAW_DATA_DIR}\n"
        "  ↳ Check BIAS_PATTERN and RAW_DATA_DIR in the Configuration cell."
    )

print(f"Found {len(bias_files)} bias frame(s).")

# Write @-list into the bias output directory
bias_list_path = os.path.join(DIR_BIAS, 'bias.list')
write_list(bias_files, bias_list_path)

# Remove stale output
if os.path.exists(MASTER_BIAS_PATH):
    os.remove(MASTER_BIAS_PATH)
    print(f"   🗑️  Removed existing {MASTER_BIAS_PATH}")

iraf.unlearn('zerocombine')
try:
    iraf.zerocombine(
        input   = f'@{bias_list_path}',
        output  = MASTER_BIAS_PATH,
        combine = BIAS_COMBINE,
        reject  = BIAS_REJECT,
        ccdtype = '',
        process = 'no',
        delete  = 'no',
        clobber = 'no',
        scale   = 'none',
        statsec = '',
        nlow    = BIAS_NLOW,
        nhigh   = BIAS_NHIGH,
        nkeep   = 1,
        mclip   = 'yes',
        lsigma  = 3.0,
        hsigma  = 3.0,
        rdnoise = RDNOISE,
        gain    = GAIN
    )
    print(f"\n✅ Master Bias saved → {MASTER_BIAS_PATH}")
except iraf.IrafError as exc:
    print(f"❌ zerocombine failed:\n   {exc}")


FileNotFoundError: No bias frames found matching '*bias*.fits' in .
  ↳ Check BIAS_PATTERN and RAW_DATA_DIR in the Configuration cell.

##  Cell 5 — Bias Correction: Science Frames

In [ ]:
# Locate science frames depending on input mode
if SCIENCE_INPUT_MODE == 'cube':
    # Slices were written to DIR_SCIENCE in Cell 3
    science_files = sorted(glob.glob(os.path.join(DIR_SCIENCE, '*_slice*.fits')))
else:  # unstacked
    science_files = find_files(UNSTACKED_PATTERN)

if not science_files:
    raise FileNotFoundError(
        "No science frames found.\n"
        f"  Mode = '{SCIENCE_INPUT_MODE}'\n"
        f"  Searched: {'DIR_SCIENCE' if SCIENCE_INPUT_MODE == 'cube' else 'RAW_DATA_DIR'}\n"
        "  ↳ Check UNSTACKED_PATTERN / CUBE_PATTERN in the Configuration cell."
    )

print(f"Found {len(science_files)} science frame(s) to bias-correct.\n")
iraf.unlearn('ccdproc')

for src in science_files:
    bname  = PREFIX_BIAS_CORR + os.path.basename(src)
    dest   = os.path.join(DIR_SCIENCE, bname)
    if os.path.exists(dest):
        os.remove(dest)
    print(f"  {os.path.basename(src):40s}  →  {bname}")
    try:
        iraf.ccdproc(
            images   = src,
            output   = dest,
            ccdtype  = '',
            interactive = 'no',
            overscan = 'no',
            trim     = 'no',
            fixpix   = 'no',
            zerocor  = 'yes',        # ← Bias subtraction ON
            darkcor  = 'no',
            flatcor  = 'no',
            illumcor = 'no',
            fringecor= 'no',
            readcor  = 'no',
            scancor  = 'no',
            zero     = MASTER_BIAS_PATH
        )
    except iraf.IrafError as exc:
        print(f"  ⚠️  ccdproc failed for {src}:\n     {exc}")

print(f"\n✅ Bias-corrected science frames saved → {DIR_SCIENCE}")


Found 12 science frame(s) to bias-correct.

  260310A_r_300s_001.fits                   →  b_260310A_r_300s_001.fits


  260310A_r_300s_002.fits                   →  b_260310A_r_300s_002.fits
  260310A_r_300s_003.fits                   →  b_260310A_r_300s_003.fits
  260310A_r_300s_004.fits                   →  b_260310A_r_300s_004.fits
  260310A_r_300s_005.fits                   →  b_260310A_r_300s_005.fits
  260310A_r_300s_006.fits                   →  b_260310A_r_300s_006.fits
  260310A_r_300s_007.fits                   →  b_260310A_r_300s_007.fits
  260310A_r_300s_008.fits                   →  b_260310A_r_300s_008.fits
  260310A_r_300s_009.fits                   →  b_260310A_r_300s_009.fits
  260310A_r_300s_010.fits                   →  b_260310A_r_300s_010.fits
  260310A_r_300s_011.fits                   →  b_260310A_r_300s_011.fits
  260310A_r_300s_012.fits                   →  b_260310A_r_300s_012.fits

✅ Bias-corrected science frames saved → /home/void/GRB_Data/r_band/reduced/science


##  Cell 6 — Bias Correction: Flat Frames

In [ ]:
flat_files = find_files(FLAT_PATTERN)

if not flat_files:
    raise FileNotFoundError(
        f"No flat frames found matching '{FLAT_PATTERN}' in {RAW_DATA_DIR}\n"
        "  ↳ Check FLAT_PATTERN and RAW_DATA_DIR in the Configuration cell."
    )

print(f"Found {len(flat_files)} flat frame(s) to bias-correct.\n")
iraf.unlearn('ccdproc')

for src in flat_files:
    bname = PREFIX_BIAS_CORR + os.path.basename(src)
    dest  = os.path.join(DIR_FLATS, bname)
    if os.path.exists(dest):
        os.remove(dest)
    print(f"  {os.path.basename(src):40s}  →  {bname}")
    try:
        iraf.ccdproc(
            images   = src,
            output   = dest,
            ccdtype  = '',
            interactive = 'no',
            overscan = 'no',
            trim     = 'no',
            fixpix   = 'no',
            zerocor  = 'yes',        # ← Bias subtraction ON
            darkcor  = 'no',
            flatcor  = 'no',
            illumcor = 'no',
            fringecor= 'no',
            readcor  = 'no',
            scancor  = 'no',
            zero     = MASTER_BIAS_PATH
        )
    except iraf.IrafError as exc:
        print(f"  ⚠️  ccdproc failed for {src}:\n     {exc}")

print(f"\n✅ Bias-corrected flat frames saved → {DIR_FLATS}")


Found 5 flat frame(s) to bias-correct.

  flat_R_1.fits                             →  b_flat_R_1.fits
  flat_R_2.fits                             →  b_flat_R_2.fits
  flat_R_3.fits                             →  b_flat_R_3.fits
  flat_R_4.fits                             →  b_flat_R_4.fits
  flat_R_5.fits                             →  b_flat_R_5.fits

✅ Bias-corrected flat frames saved → /home/void/GRB_Data/r_band/reduced/flats


##  Cell 7 — Master Flat Creation

In [ ]:
b_flat_pattern = PREFIX_BIAS_CORR + os.path.splitext(FLAT_PATTERN)[0].lstrip('*') + '*.fits'
b_flat_files   = sorted(glob.glob(os.path.join(DIR_FLATS, PREFIX_BIAS_CORR + '*.fits')))

# Filter to only bias-corrected flats (exclude any master files)
b_flat_files = [f for f in b_flat_files
                if MASTER_FLAT_NAME not in f and NORM_MASTER_FLAT_NAME not in f]

if not b_flat_files:
    raise FileNotFoundError(
        f"No bias-corrected flat frames found in {DIR_FLATS}\n"
        "  ↳ Make sure Cell 6 completed without errors."
    )

print(f"Found {len(b_flat_files)} bias-corrected flat frame(s) to combine.\n")

flat_list_path = os.path.join(DIR_FLATS, 'flat.list')
write_list(b_flat_files, flat_list_path)

if os.path.exists(MASTER_FLAT_PATH):
    os.remove(MASTER_FLAT_PATH)
    print(f"   🗑️  Removed existing {MASTER_FLAT_PATH}")

iraf.unlearn('flatcombine')
try:
    iraf.flatcombine(
        input   = f'@{flat_list_path}',
        output  = MASTER_FLAT_PATH,
        combine = FLAT_COMBINE,
        reject  = FLAT_REJECT,
        ccdtype = '',
        process = 'no',
        subsets = 'yes',
        delete  = 'no',
        clobber = 'no',
        scale   = FLAT_SCALE,
        statsec = '',
        nlow    = 0,
        nhigh   = 0,
        nkeep   = 1,
        mclip   = 'yes',
        lsigma  = FLAT_LSIGMA,
        hsigma  = FLAT_HSIGMA,
        rdnoise = RDNOISE,
        gain    = GAIN,
        snoise  = '0.0',
        pclip   = '-0.5',
        blank   = '1.0',
        mode    = 'ql'
    )
    print(f"\n✅ Master Flat saved → {MASTER_FLAT_PATH}")
except iraf.IrafError as exc:
    print(f"❌ flatcombine failed:\n   {exc}")


Found 5 bias-corrected flat frame(s) to combine.

   📝 List written: /home/void/GRB_Data/r_band/reduced/flats/flat.list  (5 entries)

✅ Master Flat saved → /home/void/GRB_Data/r_band/reduced/flats/MasterFlat.fits


##  Cell 8 — Normalise the Master Flat
> Divides the Master Flat by its mean so pixel values cluster around 1.0,
> ensuring flat-field division preserves the true flux of science frames.

In [ ]:
if os.path.exists(NORM_MASTER_FLAT_PATH):
    os.remove(NORM_MASTER_FLAT_PATH)
    print(f"   🗑️  Removed existing {NORM_MASTER_FLAT_PATH}")

print(f"Normalising {MASTER_FLAT_PATH} ...")

try:
    stat_output = iraf.imstat(MASTER_FLAT_PATH, fields='mean', format='no', Stdout=1)
    mean_val    = float(stat_output[0].strip())
    print(f"   Master Flat mean = {mean_val:.4f} ADU")

    iraf.unlearn('imarith')
    iraf.imarith(
        operand1 = MASTER_FLAT_PATH,
        op       = '/',
        operand2 = mean_val,
        result   = NORM_MASTER_FLAT_PATH
    )
    print(f"\n✅ Normalised Master Flat saved → {NORM_MASTER_FLAT_PATH}")

except iraf.IrafError as exc:
    print(f"❌ Normalisation failed:\n   {exc}")
except (IndexError, ValueError):
    print("❌ Could not parse imstat output. Check that Master Flat exists and is valid.")


Normalising /home/void/GRB_Data/r_band/reduced/flats/MasterFlat.fits ...
   Master Flat mean = 16949.1000 ADU

✅ Normalised Master Flat saved → /home/void/GRB_Data/r_band/reduced/flats/NormMasterFlat.fits


##  Cell 9 — Flat-Field Correction: Science Frames

In [ ]:
# Bias-corrected science frames are always in DIR_SCIENCE
bc_science_files = sorted(glob.glob(os.path.join(DIR_SCIENCE, PREFIX_BIAS_CORR + '*.fits')))

if not bc_science_files:
    raise FileNotFoundError(
        f"No bias-corrected science frames found in {DIR_SCIENCE}\n"
        "  ↳ Make sure Cell 5 completed without errors."
    )

print(f"Found {len(bc_science_files)} bias-corrected science frame(s) to flat-field.\n")
iraf.unlearn('ccdproc')

for src in bc_science_files:
    fname = PREFIX_FLAT_CORR + os.path.basename(src)   # e.g.  fb_b_01_12_slice001.fits
    dest  = os.path.join(DIR_SCIENCE, fname)
    if os.path.exists(dest):
        os.remove(dest)
    print(f"  {os.path.basename(src):45s}  →  {fname}")
    try:
        iraf.ccdproc(
            images   = src,
            output   = dest,
            ccdtype  = '',
            interactive = 'no',
            overscan = 'no',
            trim     = 'no',
            fixpix   = 'no',
            zerocor  = 'no',         # ← Already done in Cell 5
            darkcor  = 'no',
            flatcor  = 'yes',        # ← Flat-field correction ON
            flat     = NORM_MASTER_FLAT_PATH,
            illumcor = 'no',
            fringecor= 'no',
            readcor  = 'no',
            scancor  = 'no'
        )
    except iraf.IrafError as exc:
        print(f"  ⚠️  ccdproc failed for {src}:\n     {exc}")

print(f"\n✅ Fully-reduced science frames saved → {DIR_SCIENCE}")


Found 12 bias-corrected science frame(s) to flat-field.

  b_260310A_r_300s_001.fits                      →  fb_260310A_r_300s_001.fits
  b_260310A_r_300s_002.fits                      →  fb_260310A_r_300s_002.fits
  b_260310A_r_300s_003.fits                      →  fb_260310A_r_300s_003.fits
  b_260310A_r_300s_004.fits                      →  fb_260310A_r_300s_004.fits
  b_260310A_r_300s_005.fits                      →  fb_260310A_r_300s_005.fits
  b_260310A_r_300s_006.fits                      →  fb_260310A_r_300s_006.fits
  b_260310A_r_300s_007.fits                      →  fb_260310A_r_300s_007.fits
  b_260310A_r_300s_008.fits                      →  fb_260310A_r_300s_008.fits
  b_260310A_r_300s_009.fits                      →  fb_260310A_r_300s_009.fits
  b_260310A_r_300s_010.fits                      →  fb_260310A_r_300s_010.fits
  b_260310A_r_300s_011.fits                      →  fb_260310A_r_300s_011.fits
  b_260310A_r_300s_012.fits                      →  fb_260310A_r_300s_012.

# Cell 10 : Permanent Edge Trimming (Pre-Alignment)

In [ ]:
# =====================================================================
# 10. Permanent Edge Trimming (Pre-Alignment)
# =====================================================================
import os
import glob
import numpy as np
from astropy.io import fits

def trim_science_frames(file_pattern, trim_amount, out_prefix):
    files = sorted(glob.glob(file_pattern))
    if not files:
        print(f"⚠️ No files found matching '{file_pattern}'. Check your prefix.")
        return

    print(f"✂️ Trimming {trim_amount} pixels from all edges for {len(files)} files...")

    for file_path in files:
        dir_name = os.path.dirname(file_path)
        base_name = os.path.basename(file_path)
        out_path = os.path.join(dir_name, f"{out_prefix}{base_name}")

        with fits.open(file_path) as hdul:
            # Force 64-bit float to prevent byte-order errors later
            data = hdul[0].data.astype(np.float64) 
            header = hdul[0].header.copy()

            # PERMANENT TRIM: Slice the NumPy array [Y_start:Y_end, X_start:X_end]
            trimmed_data = data[trim_amount:-trim_amount, trim_amount:-trim_amount]

            # Update header to reflect the new smaller dimensions
            header['NAXIS1'] = trimmed_data.shape[1]
            header['NAXIS2'] = trimmed_data.shape[0]
            header['HISTORY'] = f"Trimmed {trim_amount} pixels from all edges."

            # Save the cleanly trimmed file
            fits.writeto(out_path, trimmed_data, header, overwrite=True, output_verify='silentfix')

        print(f"  [Trimmed] {base_name}  ->  {os.path.basename(out_path)}")

    print(f"\n✅ Trimming complete! Your images are now safe to align.")

# Run the trimming process
trim_science_frames(os.path.join(DIR_SCIENCE, TRIM_INPUT_WILDCARD), TRIM_PIXELS, TRIM_OUTPUT_PREFIX)

##  Cell 11 — Reduction Summary

In [ ]:
print("=" * 60)
print("         REDUCTION PIPELINE COMPLETE")
print("=" * 60)

for label, directory, pattern in [
    ("BIAS",    DIR_BIAS,    '*.fits'),
    ("FLATS",   DIR_FLATS,   '*.fits'),
    ("SCIENCE", DIR_SCIENCE, 'fb_*.fits'),
    ("TRIMMED", DIR_SCIENCE, f"{TRIM_OUTPUT_PREFIX}*.fits"),
]:
    files = sorted(glob.glob(os.path.join(directory, pattern)))
    print(f"\n📁 {label}  ({directory})")
    for f in files:
        size_kb = os.path.getsize(f) / 1024
        print(f"   {os.path.basename(f):50s}  {size_kb:8.1f} KB")
    if not files:
        print("   (empty)")

print("\n" + "=" * 60)
print("Key calibration files:")
print(f"  Master Bias         : {MASTER_BIAS_PATH}")
print(f"  Master Flat         : {MASTER_FLAT_PATH}")
print(f"  Norm. Master Flat   : {NORM_MASTER_FLAT_PATH}")
print("=" * 60)
